In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model

from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path to the dataset folder INSIDE your Google Drive.
# Example: if your Drive has "MyDrive/LungCancerProject/dataset/", use that.
DRIVE_DATASET_PATH = "/content/drive/MyDrive/LungCancerProject/dataset"

# Option A: use the Drive path directly (slower — reads over Drive mount,
# fine for small/medium datasets).
TRAIN_DIR = os.path.join(DRIVE_DATASET_PATH, "train")
VAL_DIR   = os.path.join(DRIVE_DATASET_PATH, "val")
TEST_DIR  = os.path.join(DRIVE_DATASET_PATH, "test")

# Option B (recommended for large datasets): copy the dataset from Drive
# into the local Colab runtime disk first. Local disk I/O is much faster
# than reading directly from a mounted Drive during training.
COPY_TO_LOCAL = True
LOCAL_DATASET_PATH = "/content/dataset"

if COPY_TO_LOCAL and not os.path.exists(LOCAL_DATASET_PATH):
    print("Copying dataset from Google Drive to local Colab runtime...")
    os.system(f"cp -r '{DRIVE_DATASET_PATH}' '{LOCAL_DATASET_PATH}'")
    TRAIN_DIR = os.path.join(LOCAL_DATASET_PATH, "train")
    VAL_DIR   = os.path.join(LOCAL_DATASET_PATH, "val")
    TEST_DIR  = os.path.join(LOCAL_DATASET_PATH, "test")
    print("Copy complete.")


In [ ]:
IMG_SIZE      = (150, 150)      # use (256, 256) if working with CT scan set
BATCH_SIZE    = 32
TRAIN_DIR     = "dataset/train"
VAL_DIR       = "dataset/val"
TEST_DIR      = "dataset/test"

USE_PCA       = True             # set False to skip dimensionality reduction
PCA_COMPONENTS = 100

MODEL_DIR   = "models"
REPORT_DIR  = "reports"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

RANDOM_STATE = 42

In [ ]:
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
val_gen = datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)
test_gen = datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

class_indices = train_gen.class_indices
class_names = list(class_indices.keys())
print(f"Detected classes: {class_names}")

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"          # GlobalAveragePooling -> 1280-dim feature vector
)
base_model.trainable = False   # freeze; we only use it as a fixed extractor

feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)
feature_extractor.save(os.path.join(MODEL_DIR, "mobilenetv2_feature_extractor.h5"))
feature_extractor.summary()


def extract_features(generator):
    """Run all batches of a generator through MobileNetV2 and collect
    (features, labels)."""
    generator.reset()
    steps = int(np.ceil(generator.samples / generator.batch_size))
    features = feature_extractor.predict(generator, steps=steps, verbose=1)
    labels = generator.classes[: features.shape[0]]
    return features, labels


print("\nExtracting features for TRAIN set...")
X_train, y_train = extract_features(train_gen)

print("\nExtracting features for VALIDATION set...")
X_val, y_val = extract_features(val_gen)

print("\nExtracting features for TEST set...")
X_test, y_test = extract_features(test_gen)

print(f"\nFeature shapes -> Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
le = LabelEncoder()
le.fit(list(class_indices.values()))
y_train_enc = le.transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

joblib.dump(le, os.path.join(MODEL_DIR, "label_encoder.pkl"))

In [ ]:
if USE_PCA:
    pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    X_train = pca.fit_transform(X_train)
    X_val   = pca.transform(X_val)
    X_test  = pca.transform(X_test)
    joblib.dump(pca, os.path.join(MODEL_DIR, "pca.pkl"))
    print(f"PCA applied. Explained variance retained: "
          f"{pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
xgb_clf = XGBClassifier(
    objective="multi:softprob",
    num_class=len(class_names),
    eval_metric="mlogloss",
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    tree_method="hist"
)

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=2
)

print("\nStarting XGBoost hyperparameter search...")
grid_search.fit(
    X_train, y_train_enc,
    eval_set=[(X_val, y_val_enc)],
    verbose=False
)

best_model = grid_search.best_estimator_
print(f"\nBest Parameters: {grid_search.best_params_}")

# Save the trained model
best_model.save_model(os.path.join(MODEL_DIR, "xgboost_lung_cancer_model.json"))

In [ ]:
y_pred = best_model.predict(X_test)

acc  = accuracy_score(y_test_enc, y_pred)
prec = precision_score(y_test_enc, y_pred, average="weighted")
rec  = recall_score(y_test_enc, y_pred, average="weighted")
f1   = f1_score(y_test_enc, y_pred, average="weighted")

print("\n===== TEST SET PERFORMANCE =====")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall   : {rec*100:.2f}%")
print(f"F1 Score : {f1*100:.2f}%")

report_text = classification_report(y_test_enc, y_pred, target_names=class_names)
print("\nClassification Report:\n", report_text)

with open(os.path.join(REPORT_DIR, "classification_report.txt"), "w") as f:
    f.write("===== TEST SET PERFORMANCE =====\n")
    f.write(f"Accuracy : {acc*100:.2f}%\n")
    f.write(f"Precision: {prec*100:.2f}%\n")
    f.write(f"Recall   : {rec*100:.2f}%\n")
    f.write(f"F1 Score : {f1*100:.2f}%\n\n")
    f.write(report_text)

In [ ]:
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix — MobileNetV2 + XGBoost (Approach 1)")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "confusion_matrix.png"), dpi=300)
plt.close()

print(f"\nAll artifacts saved under '{MODEL_DIR}/' and '{REPORT_DIR}/'.")
print("Training pipeline complete.")